# Reranking - Continuacion de Query Expansion

Este notebook continua a `naive_rag_lab.ipynb` y `query_expansion_lab.ipynb`. Hasta ahora el pipeline es:

**expandir pregunta en variantes -> buscar en Qdrant por cada variante -> fusionar y deduplicar (query expansion) -> prompt -> LLM**.

## El problema

La fusion de query expansion ordena los documentos usando el **score de similaridad vectorial** (coseno/dot-product entre el embedding de la pregunta y el embedding del chunk). Ese score es rapido de calcular pero es una medida *aproximada* de relevancia: mide cercania semantica en el espacio de embeddings, no si el documento realmente contiene la respuesta a la pregunta.

Esto genera dos problemas tipicos:

- Documentos con score alto pero que en realidad no responden bien la pregunta (falsos positivos).
- Documentos utiles que quedan con un score apenas por debajo del top-k y se descartan.

## La solucion: Reranking

La idea es separar el proceso en dos etapas con distinto costo/precision:

1. **Retrieval (recall alto, barato)**: con embeddings + Qdrant recuperamos un conjunto **ampliado** de candidatos (ej. 15), priorizando no perder documentos relevantes.
2. **Reranking (precision alta, mas costoso)**: un modelo mas potente evalua **cada candidato en conjunto con la pregunta** (no solo su embedding aislado) y le asigna un puntaje de relevancia real. Nos quedamos con el `top_k` final segun ese puntaje.

En este notebook implementamos el reranker usando el **mismo LLM** (con salida estructurada), que lee la pregunta y cada documento candidato y le asigna un puntaje de 0 a 10.

Esto es equivalente a lo que implementamos en el backend en:
- `src/integrations/reranker_integration.py` (interfaz)
- `src/integrations/impl/openai_reranker_integration.py` (implementacion con LLM)
- `src/integrations/impl/prompt.py` (`RERANK_INSTRUCTIONS`, `RERANK_USER_PROMPT`)
- `src/services/agent_service.py` (metodo `_rerank`, y `_retrieve_with_query_expansion` ahora usa `RETRIEVAL_CANDIDATES`)

## 0. Setup

Reutilizamos la misma configuracion que en los notebooks anteriores: variables de entorno, cliente de Qdrant, embeddings y LLM.

In [ ]:
import os
from dotenv import load_dotenv

# Cargamos el .env de 01_rag_api (un nivel arriba de notebooks/)
load_dotenv(os.path.join("..", ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
ASSISTANT_NAME = os.getenv("ASSISTANT_NAME", "Asistente")

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME")

RETRIEVAL_LIMIT = 5
RETRIEVAL_CANDIDATES = 15
QUERY_EXPANSION_COUNT = 3

print("Modelo LLM:", OPENAI_MODEL)
print("Modelo de embeddings:", OPENAI_EMBEDDING_MODEL)
print("Coleccion de Qdrant:", QDRANT_COLLECTION_NAME)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# Cliente de Qdrant apuntando a la coleccion ya existente
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

# Modelo de embeddings de OpenAI (el mismo usado al indexar los documentos)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, model=OPENAI_EMBEDDING_MODEL)

# VectorStore de LangChain: envuelve el cliente + la coleccion + el modelo de embeddings
vector_store = QdrantVectorStore(
    client=client,
    collection_name=QDRANT_COLLECTION_NAME,
    embedding=embeddings,
)

# LLM que usaremos tanto para expandir la pregunta, rerankear, como para responder
llm = ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

info = client.get_collection(QDRANT_COLLECTION_NAME)
print("Conectado a la coleccion:", QDRANT_COLLECTION_NAME)
print("Puntos indexados:", info.points_count)

In [ ]:
question = "que programas tiene?"

print("Pregunta original:", question)

## 1. Paso 1 - Recuperar un conjunto ampliado de candidatos

Repetimos el pipeline de query expansion del notebook anterior (variantes + busqueda por cada una + fusion/dedup), pero ahora pedimos `RETRIEVAL_CANDIDATES` (15) en vez de `RETRIEVAL_LIMIT` (5). La idea es priorizar **recall**: traer mas candidatos de los que finalmente vamos a usar, para dejarle al reranker la decision de cuales son realmente los mejores.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Mismas plantillas que src/integrations/impl/prompt.py
QUERY_EXPANSION_INSTRUCTIONS = (
    "Eres un asistente experto en sistemas de recuperacion de informacion (RAG). "
    "Tu tarea es reformular la pregunta del usuario en variantes alternativas que "
    "preserven exactamente el mismo significado e intencion, pero que usen palabras, "
    "sinonimos o estructuras gramaticales distintas. Esto ayuda a recuperar mas "
    "documentos relevantes de una base vectorial. No respondas la pregunta, no agregues "
    "informacion nueva y responde siempre en el mismo idioma que la pregunta original."
)

QUERY_EXPANSION_USER_PROMPT = (
    "Genera {n} reformulaciones distintas de la siguiente pregunta:\n\n"
    "Pregunta original: {question}"
)


class QueryVariants(BaseModel):
    variants: list[str] = Field(description="Lista de reformulaciones de la pregunta original")


expansion_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", QUERY_EXPANSION_INSTRUCTIONS),
        ("human", QUERY_EXPANSION_USER_PROMPT),
    ]
)

structured_llm = llm.with_structured_output(QueryVariants)
expansion_chain = expansion_prompt | structured_llm

result = expansion_chain.invoke({"question": question, "n": QUERY_EXPANSION_COUNT})
variants = result.variants[:QUERY_EXPANSION_COUNT]

queries = [question] + variants

# Retrieval por cada variante, fusionando y quedandonos con el mejor score por documento
best_by_content = {}
for q in queries:
    vector = embeddings.embed_query(q)
    docs_with_scores = vector_store.similarity_search_with_score_by_vector(vector, k=RETRIEVAL_CANDIDATES)
    for doc, score in docs_with_scores:
        key = doc.page_content
        if key not in best_by_content or score > best_by_content[key][1]:
            best_by_content[key] = (doc, score)

candidates = sorted(best_by_content.values(), key=lambda item: item[1], reverse=True)[:RETRIEVAL_CANDIDATES]

print(f"Variantes generadas: {len(variants)}")
print(f"Candidatos ampliados (antes de rerank): {len(candidates)}\n")
for doc, score in candidates:
    print(f"score={score:.4f} | source={doc.metadata.get('source_file', '')}")
    print(doc.page_content[:150], "...\n")

## 2. Paso 2 - Reranking con LLM

Ahora le pasamos al LLM la pregunta original junto con **todos los candidatos** (numerados por indice) y le pedimos que asigne un puntaje de relevancia (0-10) a cada uno. Usamos `with_structured_output` con un modelo Pydantic que devuelve una lista de `{index, score}`, para poder parsear la respuesta directamente sin texto libre.

Notar la diferencia clave respecto al retrieval: aca el LLM **lee el contenido completo** de cada documento junto con la pregunta en el mismo contexto, en lugar de comparar vectores aislados. Esto es mas costoso (una llamada al LLM con todos los candidatos) pero mucho mas preciso.

Esto es equivalente a `OpenAIRerankerIntegration.rerank` en `src/integrations/impl/openai_reranker_integration.py`, que usa las plantillas `RERANK_INSTRUCTIONS` y `RERANK_USER_PROMPT` de `src/integrations/impl/prompt.py`.

In [ ]:
# Mismas plantillas que src/integrations/impl/prompt.py
RERANK_INSTRUCTIONS = (
    "Eres un asistente experto en sistemas de recuperacion de informacion (RAG). "
    "Tu tarea es evaluar que tan relevante es cada documento candidato para responder "
    "la pregunta del usuario. Para cada documento asigna un puntaje entero de 0 a 10, "
    "donde 0 significa 'totalmente irrelevante' y 10 significa 'responde directamente "
    "la pregunta'. Basate unicamente en el contenido del documento, no en su posicion "
    "en la lista, y no agregues informacion nueva."
)

RERANK_USER_PROMPT = (
    "Pregunta: {question}\n\n"
    "Documentos candidatos:\n"
    "{candidates}\n\n"
    "Asigna un puntaje de relevancia (0-10) a cada documento, identificandolo por su indice."
)


class RankedCandidate(BaseModel):
    index: int = Field(description="Indice del documento candidato, segun el orden en que fue listado")
    score: int = Field(description="Puntaje de relevancia del documento respecto a la pregunta, de 0 a 10")


class RerankedResult(BaseModel):
    ranked: list[RankedCandidate] = Field(description="Lista de candidatos puntuados por relevancia")


rerank_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RERANK_INSTRUCTIONS),
        ("human", RERANK_USER_PROMPT),
    ]
)

reranker_llm = llm.with_structured_output(RerankedResult)
rerank_chain = rerank_prompt | reranker_llm

# Formateamos los candidatos numerados por indice, tal como espera el prompt
docs_only = [doc for doc, _score in candidates]
formatted_candidates = "\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(docs_only))

rerank_result = rerank_chain.invoke({"question": question, "candidates": formatted_candidates})

print("Puntajes asignados por el reranker:\n")
for ranked in rerank_result.ranked:
    doc = docs_only[ranked.index]
    print(f"[{ranked.index}] score_llm={ranked.score} | source={doc.metadata.get('source_file', '')}")

## 3. Paso 3 - Recortar al top-k final y comparar ordenes

Ordenamos los candidatos por el puntaje del reranker (de mayor a menor) y nos quedamos con `RETRIEVAL_LIMIT` (5) documentos finales. Comparamos ese orden contra el orden original por similaridad vectorial para ver si cambia el ranking (y potencialmente cuales documentos entran/salen del top-k final).

In [ ]:
scores_by_index = {ranked.index: ranked.score for ranked in rerank_result.ranked}

ordered_indices = sorted(
    range(len(docs_only)),
    key=lambda i: scores_by_index.get(i, 0),
    reverse=True,
)

reranked_top_k = ordered_indices[:RETRIEVAL_LIMIT]

print("Top-k por similaridad vectorial (orden original de candidates):")
for i in range(RETRIEVAL_LIMIT):
    doc = docs_only[i]
    print(f"  [{i}] score_vectorial={candidates[i][1]:.4f} | source={doc.metadata.get('source_file', '')}")

print("\nTop-k tras reranking con LLM:")
for i in reranked_top_k:
    doc = docs_only[i]
    print(f"  [{i}] score_llm={scores_by_index.get(i, 0)} | source={doc.metadata.get('source_file', '')}")

final_docs = [docs_only[i] for i in reranked_top_k]

## 4. Paso 4 - Construir el prompt e invocar al LLM

Igual que en los notebooks anteriores: formateamos el contexto (ahora el top-k reordenado por el reranker) y armamos el prompt de 3 partes (system + historial + human).

In [ ]:
formatted_context = "\n".join([f"- {doc.page_content}" for doc in final_docs])

INSTRUCTIONS = (
    "Eres un asistente útil y amigable. Responde siempre en español."
    "Tu nombre es {assistant_name}."
    "Sé conciso en tus respuestas."
)

USER_PROMPT = (
    "Contexto:\n"
    "{context}\n\n"
    "Pregunta: {question}"
)

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", INSTRUCTIONS),
        ("human", USER_PROMPT),
    ]
)

answer_chain = answer_prompt | llm

response = answer_chain.invoke(
    {
        "assistant_name": ASSISTANT_NAME,
        "question": question,
        "context": formatted_context,
    }
)

print("Pregunta:", question)
print("\nRespuesta (con reranking):\n")
print(response.content)

## 5. Funcion reusable: `rag_with_reranking`

Encapsulamos todo el pipeline (expandir -> buscar candidatos ampliados -> rerankear -> recortar top-k -> prompt -> LLM) en una sola funcion, analoga a `rag_with_query_expansion()` del notebook anterior.

In [ ]:
def rag_with_reranking(
    pregunta: str,
    n_variantes: int = QUERY_EXPANSION_COUNT,
    n_candidatos: int = RETRIEVAL_CANDIDATES,
    k: int = RETRIEVAL_LIMIT,
) -> str:
    # Paso 1: generar variantes de la pregunta
    expansion_result = expansion_chain.invoke({"question": pregunta, "n": n_variantes})
    variantes = expansion_result.variants[:n_variantes]
    queries = [pregunta] + variantes

    # Paso 2: retrieval ampliado por cada variante, fusionando por mejor score
    best_by_content = {}
    for q in queries:
        vector = embeddings.embed_query(q)
        for doc, score in vector_store.similarity_search_with_score_by_vector(vector, k=n_candidatos):
            key = doc.page_content
            if key not in best_by_content or score > best_by_content[key][1]:
                best_by_content[key] = (doc, score)

    candidatos = sorted(best_by_content.values(), key=lambda item: item[1], reverse=True)[:n_candidatos]
    docs_candidatos = [doc for doc, _score in candidatos]

    # Paso 3: reranking con LLM sobre los candidatos
    candidatos_formateados = "\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(docs_candidatos))
    rerank_res = rerank_chain.invoke({"question": pregunta, "candidates": candidatos_formateados})
    scores = {r.index: r.score for r in rerank_res.ranked}
    orden = sorted(range(len(docs_candidatos)), key=lambda i: scores.get(i, 0), reverse=True)[:k]
    docs_finales = [docs_candidatos[i] for i in orden]

    # Paso 4: prompt + LLM
    contexto = "\n".join([f"- {doc.page_content}" for doc in docs_finales])
    respuesta = answer_chain.invoke(
        {
            "assistant_name": ASSISTANT_NAME,
            "question": pregunta,
            "context": contexto,
        }
    )

    return respuesta.content


print(rag_with_reranking("¿Cuales son los requisitos de inscripcion?"))

## 6. Comparacion final: naive vs query expansion vs reranking

Corremos la misma pregunta con los tres pipelines (`naive_rag`, `rag_with_query_expansion` y `rag_with_reranking`) para comparar el contexto recuperado y la respuesta final en cada etapa de sofisticacion del RAG.

In [ ]:
def naive_rag(pregunta: str, limit: int = RETRIEVAL_LIMIT) -> str:
    vector = embeddings.embed_query(pregunta)
    docs_with_scores = vector_store.similarity_search_with_score_by_vector(vector, k=limit)
    contexto = "\n".join([f"- {doc.page_content}" for doc, _score in docs_with_scores])
    respuesta = answer_chain.invoke(
        {
            "assistant_name": ASSISTANT_NAME,
            "question": pregunta,
            "context": contexto,
        }
    )
    return respuesta.content


def rag_with_query_expansion(pregunta: str, n_variantes: int = QUERY_EXPANSION_COUNT, k: int = RETRIEVAL_LIMIT) -> str:
    expansion_result = expansion_chain.invoke({"question": pregunta, "n": n_variantes})
    variantes = expansion_result.variants[:n_variantes]
    queries = [pregunta] + variantes

    best_by_content = {}
    for q in queries:
        vector = embeddings.embed_query(q)
        for doc, score in vector_store.similarity_search_with_score_by_vector(vector, k=k):
            key = doc.page_content
            if key not in best_by_content or score > best_by_content[key][1]:
                best_by_content[key] = (doc, score)

    merged = sorted(best_by_content.values(), key=lambda item: item[1], reverse=True)[:k]
    contexto = "\n".join([f"- {doc.page_content}" for doc, _score in merged])

    respuesta = answer_chain.invoke(
        {
            "assistant_name": ASSISTANT_NAME,
            "question": pregunta,
            "context": contexto,
        }
    )

    return respuesta.content


pregunta_comparacion = "que programas tiene?"

print("=== RAG naive ===")
print(naive_rag(pregunta_comparacion))

print("\n=== RAG con query expansion ===")
print(rag_with_query_expansion(pregunta_comparacion))

print("\n=== RAG con query expansion + reranking ===")
print(rag_with_reranking(pregunta_comparacion))

<!-- TODO: Eliminar esta celda manualmente, es un duplicado de la celda 0 generado por un error al insertar. -->